# 03 · ¿Qué tan sensible es el resultado al radio?

**Objetivo:** Mostrar a directores y dueños si la competencia cercana cambia el precio de una tienda, como caso de competencia local para decisiones de expansión y de precio por tienda.

**Pregunta:** ¿Cambia la diferencia si «cerca» es 1, 2 o 3 km, o si se usa la distancia al competidor más cercano?

Importa porque «cerca» depende de cómo se mueve el cliente. Si el resultado cambia mucho con el radio, no hay un
efecto robusto. A 3 km casi no quedan tiendas sin competidor: ahí el intervalo es muy amplio.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import competencia as K
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
REPS = 500
foc = pd.read_csv(PROC / "tiendas_focales.csv")
ev = foc[foc.grupo.isin(K.EVALUABLES)].copy()
rows = []
for R in K.RADIOS:
    t = K.estimate(ev, K.comp(R), reps=REPS).iloc[0]
    rows.append({"radio_km": R, **t.to_dict(), "con": int((ev.dmin <= R).sum()), "sin": int((ev.dmin > R).sum())})
rad = pd.DataFrame(rows).set_index("radio_km")
display(rad.round(2))
ds = K.estimate(ev, K.distance_term(), reps=REPS)
print("Distancia continua: cambio del índice al duplicar la distancia al competidor más cercano")
dup = ds.loc["log_distancia"]
dup2 = {k: (np.exp(np.log1p(dup[k] / 100) * np.log(2)) - 1) * 100 for k in ["diferencia_pct", "ic90_inf", "ic90_sup"]}
print({k: round(v, 2) for k, v in dup2.items()})

,diferencia_pct,ic90_inf,ic90_sup,solido,con,sin
radio_km,,,,,,
1,0.41,0.05,0.90,True,81,136
2,0.35,-0.16,0.88,False,162,55
3,1.52,0.43,2.71,True,194,23


Distancia continua: cambio del índice al duplicar la distancia al competidor más cercano
{'diferencia_pct': np.float64(-0.26), 'ic90_inf': np.float64(-0.55), 'ic90_sup': np.float64(-0.01)}


In [2]:
rad.to_csv(PROC / "efecto_por_radio.csv", encoding="utf-8-sig")
pd.DataFrame([dup2]).to_csv(PROC / "efecto_distancia.csv", index=False, encoding="utf-8-sig")
F = "03_radio_y_distancia"
for R, r_ in rad.iterrows():
    reg(f"radio_{R}km", r_.diferencia_pct, [r_.ic90_inf, r_.ic90_sup], n=r_.con + r_.sin, fuente=F)
    reg(f"radio_{R}km_sin", r_.sin, fuente=F)
reg("duplicar_distancia", dup2["diferencia_pct"], [dup2["ic90_inf"], dup2["ic90_sup"]], fuente=F)
display(Markdown(
    f"**Conclusión.** A 1 km la diferencia es {rad.loc[1, 'diferencia_pct']:+.2f}% (IC90 {rad.loc[1, 'ic90_inf']:+.2f} a {rad.loc[1, 'ic90_sup']:+.2f}), a 2 km "
    f"{rad.loc[2, 'diferencia_pct']:+.2f}% y a 3 km {rad.loc[3, 'diferencia_pct']:+.2f}% (con solo {int(rad.loc[3, 'sin'])} tiendas sin competidor). Duplicar "
    f"la distancia al competidor más cercano cambia el índice {dup2['diferencia_pct']:+.2f}%. En ningún radio aparece un descuento por competencia."))

**Conclusión.** A 1 km la diferencia es +0.41% (IC90 +0.05 a +0.90), a 2 km +0.35% y a 3 km +1.52% (con solo 23 tiendas sin competidor). Duplicar la distancia al competidor más cercano cambia el índice -0.26%. En ningún radio aparece un descuento por competencia.